In [10]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

data = np.load("../../FluidGPT_validation/tensordata/amira__0__AR-d2__.npz", allow_pickle=True)
data2 = np.load("../../FluidGPT_validation/tensordata/amira__0__AR-d3__.npz", allow_pickle=True)
data3 = np.load("../../FluidGPT_validation/tensordata/amira__0__FM-d2__.npz", allow_pickle=True)
data4 = np.load("../../FluidGPT_validation/tensordata/amira__0__FM-d3__.npz", allow_pickle=True)

datasetlist = ['pdebench-comp'] #['amira', 'pdebench-comp', 'pdebench-incomp', 'pdegym-gauss', 'pdegym-sines', 'pdegym-bb', 'pdegym-pwc', 'pdegym-svs', 'pdegym-sl']
trajlist = [0] #,1,2,3,4,5,6,7,8,9]
models = ['AR-d2', 'AR-d3', 'FM-d2', 'FM-d3']
base_dir = "../../FluidGPT_validation/tensordata/"

# See what arrays are stored
print(data.files)
ground_truth = data['traj_pred']
prediction = data['traj_true']
print(ground_truth.shape, prediction.shape)
print(ground_truth.dtype, prediction.dtype)

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 150  # value in MB, default is 20

['label', 'model_type', 'dsplit_idx', 'traj_idx', 'actual_traj_idx', 'traj_pred', 'traj_true', 'rae_errors', 'rrmse_errors', 'cfg']
(1, 201, 2, 128, 128) (1, 201, 2, 128, 128)
float32 float32


## DUE TO A BUG THE GROUND TRUTH AND PREDICTIONS ARE REVERSED!!!

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
#from scipy.ndimage import uniform_filter1d
from scipy.ndimage import gaussian_filter1d
from IPython.display import HTML

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
QUANTITY = 'radial'   # 'vorticity' or 'radial'
CMAP = 'viridis'
CMAP_ERROR = 'magma'
FPS_INTERVAL = 100

SMOOTH_WINDOW = 14      # frames; odd number, larger = smoother/slower to react
CLIM_PERCENTILE = 99   # use 99th percentile instead of true max, to avoid outlier-driven jitter
GAUSSIAN_SIGMA = 3.0

# ------------------------------------------------------------------
# FIELD COMPUTATION
# ------------------------------------------------------------------
def compute_vorticity(traj):
    vx, vy = traj[:, 0], traj[:, 1]
    dvy_dx = np.gradient(vy, axis=2)
    dvx_dy = np.gradient(vx, axis=1)
    return dvy_dx - dvx_dy

def compute_speed(traj):
    vx, vy = traj[:, 0], traj[:, 1]
    return np.sqrt(vx**2 + vy**2)

def compute_field(traj, quantity):
    if quantity == 'vorticity':
        return compute_vorticity(traj)
    elif quantity == 'radial':
        return compute_speed(traj)
    else:
        raise ValueError("quantity must be 'vorticity' or 'radial'")

for dataset in datasetlist:
    for traj in trajlist:
        raw_data = []
        for model in models:
            file_dir = f"{base_dir}{dataset}__{traj}__{model}__.npz"
            raw_data.append(np.load(file_dir, allow_pickle=True))
        data = raw_data[0]
        ground_truth = data['traj_pred'].squeeze(0)
        print(ground_truth.shape, ground_truth.dtype)
        data2 = raw_data[1]
        data3 = raw_data[2]
        data4 = raw_data[3]

        datasets = {
            'AR-d2': data,
            'AR-d3': data2,
            'FM-d2': data3,
            'FM-d3': data4,
        }

        field_gt = compute_field(ground_truth, QUANTITY)
        field_pred = {k: compute_field(v['traj_true'][0], QUANTITY) for k, v in datasets.items()}
        field_err = {k: np.abs(field_pred[k] - field_gt) for k in datasets}

        n_frames = field_gt.shape[0]
        print(f"Number of frames: {n_frames}")
        quantity_label = 'Vorticity' if QUANTITY == 'vorticity' else 'Velocity Magnitude'

        # ------------------------------------------------------------------
        # PRECOMPUTE + SMOOTH COLOR LIMITS FOR THE WHOLE TRAJECTORY
        # ------------------------------------------------------------------
        def raw_main_clim(frame):
            """Percentile-based vmin/vmax across GT + all predictions, for one frame."""
            frame_stack = [field_gt[frame]] + [field_pred[k][frame] for k in datasets]
            if QUANTITY == 'vorticity':
                m = max(np.percentile(np.abs(f), CLIM_PERCENTILE) for f in frame_stack)
                return -m, m
            else:
                lo = min(np.percentile(f, 100 - CLIM_PERCENTILE) for f in frame_stack)
                hi = max(np.percentile(f, CLIM_PERCENTILE) for f in frame_stack)
                return lo, hi

        # Build raw (unsmoothed) per-frame arrays, shape (n_frames, 2) -> [:,0]=vmin, [:,1]=vmax
        raw_main = np.array([raw_main_clim(f) for f in range(n_frames)])

        # Smooth each column (vmin, vmax) independently across time
        #smooth_vmin = uniform_filter1d(raw_main[:, 0], size=SMOOTH_WINDOW, mode='nearest')
        #smooth_vmax = uniform_filter1d(raw_main[:, 1], size=SMOOTH_WINDOW, mode='nearest')
        smooth_vmin = gaussian_filter1d(raw_main[:, 0], sigma=GAUSSIAN_SIGMA, mode='nearest')
        smooth_vmax = gaussian_filter1d(raw_main[:, 1], sigma=GAUSSIAN_SIGMA, mode='nearest')

        # Error panels currently share the same scale as the main field (per your last version)
        smooth_err_vmin = smooth_vmin
        smooth_err_vmax = smooth_vmax

        # Initial (frame 0) scales, used to set up the plots before animating
        vmin, vmax = smooth_vmin[0], smooth_vmax[0]
        err_vmin, err_vmax = smooth_err_vmin[0], smooth_err_vmax[0]

        # ------------------------------------------------------------------
        # DARK-BACKGROUND / TRANSPARENT STYLING
        # ------------------------------------------------------------------
        TEXT_COLOR = 'white'

        def style_axis(ax, emphasize=False):
            ax.set_facecolor('none')
            ax.patch.set_alpha(0.0)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_color(TEXT_COLOR)
                spine.set_alpha(0.9 if emphasize else 0.4)
                spine.set_linewidth(1.5 if emphasize else 1.0)

        # ------------------------------------------------------------------
        # LAYOUT
        # ------------------------------------------------------------------
        fig = plt.figure(figsize=(16, 8))
        fig.patch.set_alpha(0.0)

        gs = gridspec.GridSpec(
            2, 5, figure=fig,
            width_ratios=[0.6, 1, 1.3, 1, 0.6],
            wspace=0.05, hspace=0.15,
            top=0.90, bottom=0.16, left=0.02, right=0.98
        )

        layout = {
            'AR-d2': (0, 'L'),
            'FM-d2': (1, 'L'),
            'AR-d3': (0, 'R'),
            'FM-d3': (1, 'R'),
        }

        pred_col = {'L': 1, 'R': 3}
        err_col  = {'L': 0, 'R': 4}

        pred_axes, pred_images = {}, {}
        err_axes, err_images = {}, {}

        for label, (r, side) in layout.items():
            ax_p = fig.add_subplot(gs[r, pred_col[side]])
            im_p = ax_p.imshow(field_pred[label][0], cmap=CMAP, vmin=vmin, vmax=vmax, origin='lower')
            ax_p.set_title(label, fontsize=11, color=TEXT_COLOR, fontweight='bold')
            style_axis(ax_p)
            pred_axes[label] = ax_p
            pred_images[label] = im_p

            ax_e = fig.add_subplot(gs[r, err_col[side]])
            im_e = ax_e.imshow(field_err[label][0], cmap=CMAP_ERROR, vmin=err_vmin, vmax=err_vmax, origin='lower')
            ax_e.set_title(f"{label} |error|", fontsize=8, color=TEXT_COLOR, alpha=0.85)
            style_axis(ax_e)

            pos = ax_e.get_position()
            shrink = 1
            shift_frac = 0.1
            new_width = pos.width * shrink
            new_height = pos.height * shrink
            new_y0 = pos.y0 + (pos.height - new_height) / 2
            if side == 'L':
                new_x0 = pos.x0 + pos.width * shift_frac
            else:
                new_x0 = pos.x1 - new_width - pos.width * shift_frac
            ax_e.set_position([new_x0, new_y0, new_width, new_height])

            err_axes[label] = ax_e
            err_images[label] = im_e

        ax_gt = fig.add_subplot(gs[:, 2])
        im_gt = ax_gt.imshow(field_gt[0], cmap=CMAP, vmin=vmin, vmax=vmax, origin='lower')
        ax_gt.set_title('Ground Truth', fontsize=13, color=TEXT_COLOR, fontweight='bold')
        style_axis(ax_gt, emphasize=True)

        # ------------------------------------------------------------------
        # COLORBARS
        # ------------------------------------------------------------------
        cbar_width = 0.35
        cbar_height = 0.025
        cbar_y = 0.06
        gap = 0.06

        total_span = 2 * cbar_width + gap
        left_start = (1 - total_span) / 2

        cbar_ax_main = fig.add_axes([left_start, cbar_y, cbar_width, cbar_height])
        cbar_ax_main.patch.set_alpha(0.0)
        cbar_main = fig.colorbar(im_gt, cax=cbar_ax_main, orientation='horizontal', label=quantity_label)
        cbar_main.ax.xaxis.label.set_color(TEXT_COLOR)
        cbar_main.ax.tick_params(colors=TEXT_COLOR)
        cbar_main.outline.set_edgecolor(TEXT_COLOR)

        cbar_ax_err = fig.add_axes([left_start + cbar_width + gap, cbar_y, cbar_width, cbar_height])
        cbar_ax_err.patch.set_alpha(0.0)
        cbar_err = fig.colorbar(err_images['FM-d3'], cax=cbar_ax_err, orientation='horizontal', label='|error|')
        cbar_err.ax.xaxis.label.set_color(TEXT_COLOR)
        cbar_err.ax.tick_params(colors=TEXT_COLOR, labelsize=8)
        cbar_err.outline.set_edgecolor(TEXT_COLOR)

        suptitle = fig.suptitle(f"{quantity_label} at t = 0", fontsize=16, color=TEXT_COLOR)

        # ------------------------------------------------------------------
        # ANIMATION — uses precomputed, smoothed clim arrays (no live recompute)
        # ------------------------------------------------------------------
        def update(frame):
            f_vmin, f_vmax = smooth_vmin[frame], smooth_vmax[frame]
            f_err_vmin, f_err_vmax = smooth_err_vmin[frame], smooth_err_vmax[frame]

            for label in layout:
                pred_images[label].set_data(field_pred[label][frame])
                pred_images[label].set_clim(f_vmin, f_vmax)

                err_images[label].set_data(field_err[label][frame])
                err_images[label].set_clim(f_err_vmin, f_err_vmax)

            im_gt.set_data(field_gt[frame])
            im_gt.set_clim(f_vmin, f_vmax)

            cbar_main.update_normal(im_gt)
            cbar_err.update_normal(err_images['FM-d3'])

            suptitle.set_text(f"{quantity_label} at t = {frame}")
            return list(pred_images.values()) + list(err_images.values()) + [im_gt, suptitle]

        anim = FuncAnimation(fig, update, frames=n_frames, interval=FPS_INTERVAL, blit=False)
        plt.close(fig)

        HTML(anim.to_jshtml())

        fig.patch.set_facecolor('#12161b')
        fig.patch.set_alpha(1.0)
        anim.save(f'../../FluidGPT_validation/animations/anim_{quantity_label}_{dataset}_{traj}.mp4', writer='ffmpeg', fps=8, dpi=150)
        print(f"Saved animation for dataset '{dataset}', trajectory {traj} as 'anim_{quantity_label}_{dataset}_{traj}.mp4'")

(21, 2, 128, 128) float32
Number of frames: 21
Saved animation for dataset 'pdebench-comp', trajectory 0 as 'anim_Velocity Magnitude_pdebench-comp_0.mp4'
